# Triply Robust Panel (TROP) Estimator — Stata 教程

本教程演示 **TROP Stata 包** 的完整功能。

TROP（Athey, Imbens, Qu & Viviano, 2025）结合三个稳健性组件：

1. **核范数正则化因子模型**：通过矩阵补全估计交互固定效应
2. **指数距离单位权重**：$\omega_j = \exp(-\lambda_{\text{unit}} \times \text{dist}(j,i))$，基于控制期结果差异加权
3. **指数时间衰减权重**：$\theta_s = \exp(-\lambda_{\text{time}} \times |s-t|)$，按时间邻近度加权

**适用场景**：
- 数据中可能存在因子结构的未观测时变混杂
- 标准 DiD 或 SDID 因潜在因子而产生偏误
- 有足够长的处理前期来估计因子

**本教程涵盖**：
1. 数据生成与探索
2. 基本 TROP 估计（LOOCV 调优）
3. 调优参数理解
4. 因子结构分析
5. 诊断工具（estat 命令族）
6. 预测功能
7. Twostep vs Joint 方法对比
8. Bootstrap 推断
9. 真实数据示例（CPS）

In [1]:
python:
import numpy as np
import pandas as pd
from sfi import Data, Macro

n_units = 30
n_pre = 6
n_post = 3
n_treated = 6
n_factors = 2
treatment_effect = 2.0
factor_strength = 1.5
treated_loading_shift = 0.5
unit_fe_sd = 1.0
noise_sd = 0.5
seed = 42

rng = np.random.default_rng(seed)
n_control = n_units - n_treated
n_periods = n_pre + n_post

F = rng.normal(0, 1, (n_periods, n_factors))
Lambda = rng.normal(0, 1, (n_factors, n_units))
Lambda[:, :n_treated] += treated_loading_shift
alpha = rng.normal(0, unit_fe_sd, n_units)
alpha[:n_treated] += 1.0
beta = np.linspace(0, 2, n_periods)

records = []
for i in range(n_units):
    is_ever_treated = i < n_treated
    for t in range(n_periods):
        post = t >= n_pre
        y = 10.0 + alpha[i] + beta[t]
        y += factor_strength * (Lambda[:, i] @ F[t, :])
        effect = 0.0
        if is_ever_treated and post:
            effect = treatment_effect
            y += effect
        y += rng.normal(0, noise_sd)
        records.append({
            "unit": i,
            "period": t,
            "outcome": y,
            "treated": int(is_ever_treated and post),
            "treat": int(is_ever_treated),
            "true_effect": effect,
        })

df = pd.DataFrame(records)
n_obs = len(df)
Data.setObsTotal(n_obs)
Data.addVarDouble("unit")
Data.addVarDouble("period")
Data.addVarDouble("outcome")
Data.addVarDouble("treated")
Data.addVarDouble("treat")
Data.addVarDouble("true_effect")

for i in range(n_obs):
    Data.storeAt("unit",     i, df.iloc[i]["unit"])
    Data.storeAt("period",   i, df.iloc[i]["period"])
    Data.storeAt("outcome",  i, df.iloc[i]["outcome"])
    Data.storeAt("treated",  i, df.iloc[i]["treated"])
    Data.storeAt("treat",    i, df.iloc[i]["treat"])
    Data.storeAt("true_effect", i, df.iloc[i]["true_effect"])

print("Dataset: %d obs" % n_obs)
print("Treated units: %d" % n_treated)
print("Control units: %d" % n_control)
print("Pre-periods: %d" % n_pre)
print("Post-periods: %d" % n_post)
print("True ATT: %.1f" % treatment_effect)
print("True factors: %d" % n_factors)
end


Running /Applications/Stata/profile.do ...


(41 lines skipped)
r(7102);


  File "<stdin>", line 1
    for i in range(n_units):
                            ^
IndentationError: expected an indented block after 'for' statement on line 1


In [ ]:
describe
summarize outcome treated treat true_effect
tabulate treated treat
table period treated, statistic(mean outcome) nformat(%9.3f)

## 1. 何时使用 TROP

当以下情况时，考虑使用 TROP：
- 你怀疑数据中存在**因子结构**（如经济周期、区域冲击）
- **未观测混杂因素**以不同方式随时间影响各单位
- 标准平行趋势假设可能因共同因子而被违反
- 你有**足够长的处理前期**来估计因子

TROP 与 SDID 的关键区别在于：TROP 显式建模并移除交互固定效应（因子贡献），然后再计算处理效应。

In [2]:
adopath + "../../trop_stata/ado"

trop outcome treated, panelvar(unit) timevar(period) method(twostep) verbose

  [1]  (BASE)      "/Applications/Stata/ado/base/"
  [2]  (SITE)      "/Applications/Stata/ado/site/"
  [3]              "."
  [4]  (PERSONAL)  "/Applications/Stata/personal/"
  [5]  (PLUS)      "/Applications/Stata/plus/"
  [6]  (OLDPLACE)  "~/ado/"
  [7]              "../../trop_stata/ado"
command trop is unrecognized
r(199);


In [3]:
ereturn list

display "True ATT: 2.0000"
display "Est. ATT: " %12.6f e(att)
display "Bias:     " %12.6f e(att) - 2.0
display ""
display "Selected tuning parameters:"
display "  lambda_time: " %8.4f e(lambda_time)
display "  lambda_unit: " %8.4f e(lambda_unit)
display "  lambda_nn:   " %8.4f e(lambda_nn)

True ATT: 2.0000
Est. ATT:            .
Bias:            .

Selected tuning parameters:
 lambda_time:        .
 lambda_unit:        .
 lambda_nn:        .


## 2. 理解调优参数

TROP 通过留一交叉验证（LOOCV）选择三个调优参数：

### λ_time（时间衰减）
控制对接近处理时点的期数赋予多少权重：
- **λ_time = 0**：所有处理前期等权
- **λ_time > 0**：更多权重给近期的处理前期

### λ_unit（单位距离）
控制对相似控制单位赋予多少权重：
- **λ_unit = 0**：所有控制单位等权
- **λ_unit > 0**：更多权重给处理前轨迹相似的控制单位

单位间距离计算为结果变量的均方根差异，仅使用双方均未受处理的期数，且排除目标期 $t$（论文公式 3）。

### λ_nn（核范数）
控制因子模型的秩：
- **λ_nn = 0**：无正则化（满秩）
- **λ_nn > 0**：鼓励低秩因子结构

In [ ]:
trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) fixedlambda(1.0 1.0 0.1)

display "Fixed lambda results:"
display "  ATT = " %12.6f e(att)
display "  lambda_time = " %8.4f e(lambda_time)
display "  lambda_unit = " %8.4f e(lambda_unit)
display "  lambda_nn   = " %8.4f e(lambda_nn)

In [ ]:
quietly trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) fixedlambda(1.0 1.0 0.0)
local att_nn0 = e(att)

quietly trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) fixedlambda(1.0 1.0 0.1)
local att_nn01 = e(att)

quietly trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) fixedlambda(1.0 1.0 1.0)
local att_nn1 = e(att)

display ""
display "Effect of nuclear norm penalty (lambda_nn):"
display "{hline 55}"
display %10s "lambda_nn" %14s "ATT" %14s "Bias"
display "{hline 55}"
display %10.1f 0.0 %14.4f `att_nn0'  %14.4f `att_nn0' - 2.0
display %10.1f 0.1 %14.4f `att_nn01' %14.4f `att_nn01' - 2.0
display %10.1f 1.0 %14.4f `att_nn1'  %14.4f `att_nn1' - 2.0
display "{hline 55}"

In [ ]:
trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) ///
    lambda_time_grid(0.0 0.5 1.0 2.0) ///
    lambda_unit_grid(0.0 0.5 1.0 2.0) ///
    lambda_nn_grid(0.0 0.01 0.1 1.0) ///
    verbose

display ""
display "Custom grid selected hyperparameters:"
display "  lambda_time = " %8.4f e(lambda_time)
display "  lambda_unit = " %8.4f e(lambda_unit)
display "  lambda_nn   = " %8.4f e(lambda_nn)
display "  ATT         = " %12.6f e(att)

## 3. 因子结构分析

TROP 估计一个低秩因子矩阵 $L$，捕获交互固定效应。我们可以通过 `estat factors` 检查该结构。

In [ ]:
quietly trop outcome treated, panelvar(unit) timevar(period) method(twostep)

estat factors

In [ ]:
estat summarize

## 4. 单位效应和时间效应

TROP 同时估计传统的单位固定效应 $\alpha_i$ 和时间固定效应 $\beta_t$。可以通过 `predict` 命令提取。

In [ ]:
predict alpha_hat, alpha
tabstat alpha_hat, by(unit) statistics(mean) columns(statistics)

predict beta_hat, beta
tabstat beta_hat, by(period) statistics(mean) columns(statistics)

## 5. TROP vs DID 比较

通过设置 `fixedlambda(0 0 1e10)` 可以模拟标准 DID 估计器（无因子调整、等权）。与完整 TROP 对比，可以看到因子调整的重要性。

In [ ]:
quietly trop outcome treated, panelvar(unit) timevar(period) method(twostep)
local att_trop = e(att)
local lt_trop = e(lambda_time)
local lu_trop = e(lambda_unit)
local ln_trop = e(lambda_nn)

quietly trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) fixedlambda(0 0 1e10)
local att_did = e(att)

display ""
display "TROP vs DID (True ATT = 2.0)"
display "{hline 60}"
display %20s "Estimator" %14s "ATT" %14s "Bias"
display "{hline 60}"
display %20s "TROP (LOOCV)" %14.4f `att_trop' %14.4f `att_trop' - 2.0
display %20s "DID (simulated)" %14.4f `att_did' %14.4f `att_did' - 2.0
display "{hline 60}"
display ""
display "TROP selected hyperparameters:"
display "  lambda_time = " %8.4f `lt_trop'
display "  lambda_unit = " %8.4f `lu_trop'
display "  lambda_nn   = " %8.4f `ln_trop'

## 6. Twostep vs Joint 方法

TROP 支持两种估计方法：

**Twostep**（`method(twostep)`，默认）：
- 对应论文 Algorithm 2
- 为每个处理观测计算观测特定权重，拟合单独模型后取平均
- 更灵活，允许异质处理效应
- 计算量较大

**Joint**（`method(joint)`）：
- 加权最小二乘，估计单一标量处理效应 τ
- 权重只计算一次（到处理块中心的距离）
- 更快，但假设同质处理效应

In [ ]:
quietly trop outcome treated, panelvar(unit) timevar(period) method(twostep)
local att_twostep = e(att)
local lt_twostep = e(lambda_time)
local lu_twostep = e(lambda_unit)
local ln_twostep = e(lambda_nn)

quietly trop outcome treated, panelvar(unit) timevar(period) method(joint)
local att_joint = e(att)
local lt_joint = e(lambda_time)
local lu_joint = e(lambda_unit)
local ln_joint = e(lambda_nn)

display ""
display "Twostep vs Joint (True ATT = 2.0)"
display "{hline 65}"
display %15s "Method" %12s "ATT" %12s "Bias" %10s "l_time" %10s "l_unit" %10s "l_nn"
display "{hline 65}"
display %15s "Twostep" %12.4f `att_twostep' %12.4f `att_twostep'-2.0 ///
    %10.2f `lt_twostep' %10.2f `lu_twostep' %10.2f `ln_twostep'
display %15s "Joint" %12.4f `att_joint' %12.4f `att_joint'-2.0 ///
    %10.2f `lt_joint' %10.2f `lu_joint' %10.2f `ln_joint'
display "{hline 65}"

## 7. Bootstrap 推断

TROP 使用**单位层分层块 bootstrap**进行方差估计（论文 Algorithm 3）。控制组和处理组单位分别重抽样以保持处理比例。通过 `bootstrap()` 选项指定重复次数。

In [ ]:
trop outcome treated, panelvar(unit) timevar(period) ///
    method(twostep) bootstrap(200) seed(42)

display ""
display "Bootstrap inference results:"
display "  ATT      = " %12.6f e(att)
display "  SE       = " %12.6f e(se)
display "  t-stat   = " %12.4f e(tstat)
display "  p-value  = " %12.4f e(pvalue)
display "  95% CI   = [" %8.4f e(ci_lower) ", " %8.4f e(ci_upper) "]"
display ""
display "True ATT = 2.0 in 95% CI?"
display cond(e(ci_lower) <= 2.0 & 2.0 <= e(ci_upper), "  -> Yes", "  -> No")

In [ ]:
estat bootstrap

In [ ]:
estat vce

## 8. 预测功能

`predict` 命令可以提取多种估计量：

| 选项 | 含义 |
|------|------|
| `y0` | 反事实结果 $\hat{Y}(0)$（无处理时的预测值） |
| `te` | 处理效应 $\hat{\tau}_{it}$（仅处理观测有值） |
| `residuals` | 残差 $e_{it} = Y_{it} - \hat{Y}(0)_{it} - \hat{\tau}_{it} \cdot D_{it}$ |
| `alpha` | 单位固定效应 $\hat{\alpha}_i$ |
| `beta` | 时间固定效应 $\hat{\beta}_t$ |

In [ ]:
quietly trop outcome treated, panelvar(unit) timevar(period) method(twostep)

capture drop y0_hat te_hat resid_hat

predict y0_hat, y0
predict te_hat, te
predict resid_hat, residuals

display ""
display "Prediction summary:"
summarize y0_hat te_hat resid_hat

display ""
display "Treatment effect distribution (treated obs only):"
summarize te_hat if treated == 1

display ""
display "Residual mean (should be near 0):"
summarize resid_hat, detail

## 9. 权重与 LOOCV 诊断

TROP 的核心在于自适应权重。`estat weights` 展示单位权重和时间权重的分布，帮助理解哪些控制单位和哪些时期对估计贡献最大。`estat loocv` 展示 LOOCV 超参数选择的详细过程。

In [ ]:
estat weights

In [ ]:
estat loocv

## 10. 附加说明

TROP 的 Stata 实现采用 Rust 计算后端，保证高性能和数值精度。所有核心算法（距离计算、权重生成、LOOCV、SVD 求解、Bootstrap）均在 Rust 层完成，Mata 层负责数据验证和结果存储。

In [ ]:
display "TROP Stata package architecture:"
display "  Stata ADO -> Mata interface -> C bridge -> Rust core"
display ""
display "Key features:"
display "  - Nuclear-norm regularized SVD estimation"
display "  - Coordinate-descent LOOCV for hyperparameter selection"  
display "  - Stratified unit block bootstrap for inference"

## 11. 真实数据示例 — CPS 最低工资数据

使用 Current Population Survey (CPS) 州级面板数据，估计最低工资对对数工资的影响。该数据集来自 Arkhangelsky et al. (2021) 的 Synthetic DiD 论文，包含 50 个州 1979–2019 年的年度面板。

处理变量 `min_wage` 为二值指标，标记各州是否实施了高于联邦标准的最低工资政策。

In [ ]:
preserve

import delimited "../../data/CPS.csv", delimiter(";") clear

generate byte min_wage_d = (min_wage == "TRUE")
encode state, generate(state_id)

describe
summarize log_wage min_wage_d
tabulate min_wage_d

trop log_wage min_wage_d, panelvar(state_id) timevar(year) ///
    method(twostep) bootstrap(100) seed(42) verbose

display ""
display "CPS real data results:"
display "  ATT      = " %12.6f e(att)
display "  SE       = " %12.6f e(se)
display "  t-stat   = " %12.4f e(tstat)
display "  p-value  = " %12.4f e(pvalue)
display "  95% CI   = [" %8.4f e(ci_lower) ", " %8.4f e(ci_upper) "]"
display ""
display "Selected hyperparameters:"
display "  lambda_time = " %8.4f e(lambda_time)
display "  lambda_unit = " %8.4f e(lambda_unit)
display "  lambda_nn   = " %8.4f e(lambda_nn)

restore

## 12. 总结

### 关键要点

1. **TROP 通过三重稳健性处理因子混杂**：核范数正则化因子模型 + 指数距离单位权重 + 指数时间衰减权重，在存在交互固定效应时比标准 DiD/SDID 更稳健。

2. **LOOCV 自动调优**：三个超参数 $(\lambda_{\text{time}}, \lambda_{\text{unit}}, \lambda_{\text{nn}})$ 通过留一交叉验证自动选择，无需手动调参。也可通过 `fixedlambda()` 或自定义网格控制。

3. **两种估计方法**：
   - `twostep`：更灵活，允许异质处理效应，适合探索性分析
   - `joint`：更快，假设同质效应，适合大样本

4. **完整推断工具链**：`bootstrap()` 提供标准误、置信区间和 p 值；`estat` 命令族提供因子分析、权重诊断、LOOCV 诊断等。

5. **高性能 Rust 后端**：核心计算在 Rust 中完成，保证数值精度和计算效率。

### 何时选择 TROP vs SDID

| 场景 | 推荐方法 |
|------|----------|
| 怀疑存在因子结构 | TROP |
| 处理前期较长（≥5期） | TROP |
| 标准平行趋势可能被违反 | TROP |
| 处理前期较短（2-3期） | SDID |
| 无因子混杂的简单设定 | DiD / SDID |

### 命令速查

```stata
* 基本估计
trop y d, panelvar(id) timevar(t) method(twostep)

* 带 bootstrap 推断
trop y d, panelvar(id) timevar(t) bootstrap(200) seed(42)

* 固定超参数
trop y d, panelvar(id) timevar(t) fixedlambda(1.0 1.0 0.1)

* 诊断命令
estat factors      // 因子矩阵 SVD
estat summarize    // 样本摘要
estat weights      // 权重分布
estat loocv        // LOOCV 诊断
estat bootstrap    // Bootstrap 分布
estat vce          // 方差协方差矩阵

* 预测
predict y0_hat, y0         // 反事实结果
predict te_hat, te         // 处理效应
predict resid, residuals   // 残差
predict alpha_hat, alpha   // 单位固定效应
predict beta_hat, beta     // 时间固定效应
```